In [1]:
## this is the script that filters the data for modelling using MOGONET

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.feature_selection import VarianceThreshold

In [2]:
# read data
log_counts = pd.read_csv('log_corrected_counts_trauma.tsv', sep = '\t', index_col = 0)
clin = pd.read_csv('/Users/smasarone/Documents/jupyter_notebooks_copy2/clin_rnaseq.csv', header = 0, index_col = 0)
log_counts = log_counts.T
log_counts = log_counts.filter(items = clin.index, axis = 0)

# import also proteomics
prot = pd.read_csv('/Users/smasarone/Documents/jupyter_notebooks_copy2/raw_data_proteins_filtered.csv',header = 0, index_col = 0)
prot_list = pd.read_csv('/Users/smasarone/Documents/jupyter_notebooks_copy2/prot_filtered.csv', header = 0, index_col=0)
prot = prot.filter(prot_list.iloc[:,0], axis = 1)
prot = prot.drop(labels = [1578], axis = 0)

new_idx_p = []
for i in prot.index:
    new_idx_p.append('S'+str(i))
prot.index = new_idx_p

# filter clin and counts by prot
counts = log_counts.filter(items = prot.index, axis = 0)
clin = clin.filter(items = prot.index, axis = 0)
prot = prot.filter(items = clin.index, axis =0)
print(clin.shape)
print(counts.shape)
print(prot.shape)

(412, 49)
(412, 20782)
(412, 4979)


### FILTER COUNTS DATA FIRST

In [21]:
### Check ANOVA for the two groups counts (ISS >25 and below)
from scipy.stats import f_oneway
import random

above_idx = clin[clin['iss']>=25].index
below_idx = clin[clin['iss']<25].index
assert len(above_idx)+len(below_idx)==412

# split before calculating the anova
y = [1 if i >=25 else 0 for i in clin['iss']]

# generate 5 random seeds to split the data
random_seeds=[]

for i in range(0,5):
    n = random.randint(1,300)
    random_seeds.append(n)
print(random_seeds)

random_seeds = [238, 88, 99, 47, 7] # have to save the numbers otherwise it's always random
for seed in random_seeds:
    X_train, X_test, y_train, y_test = train_test_split(counts, y, random_state=i)
    feature_counts = counts.columns

    above_df = X_train.filter(items = above_idx, axis = 0)
    below_df = X_train.filter(items = below_idx, axis = 0)

    # calculate the anova on the train only
    f_list=[]
    p_list=[]
    for i in above_df.columns:
        F, p = f_oneway(above_df.loc[:,i], below_df.loc[:,i])
        f_list.append(F)
        p_list.append(p)


    threshold = 0.05/len(above_df.columns) 

    relevant=[]
    for i, z in enumerate(p_list):
        if z<threshold:
            relevant.append(i)
    print(len(relevant))

    # print the first few genes that have been selected as different
    # make a table 
    df_anova = pd.DataFrame(np.array(p_list)[relevant], columns = ['p-values'], index = counts.columns[relevant])
    df_anova['f-values'] = np.array(f_list)[relevant]
    we_want = df_anova.sort_values(by ='f-values', ascending = False)[0:200].index

    X_train_=X_train.loc[:, we_want]
    X_test_= X_test.loc[:, we_want]
    print(X_train_.shape)
    print(X_test_.shape)

    final_col = counts.columns[relevant]
    np.savetxt(str(seed)+'_2_featname.csv', we_want, delimiter=',', fmt = '%s')
    np.savetxt(str(seed)+'_2_tr.csv', X_train_, delimiter=',')
    np.savetxt(str(seed)+'_2_te.csv', X_test_, delimiter=',')
    
    
    ## PROT
    X_train, X_test, y_train, y_test = train_test_split(prot, y, random_state=seed)
    feature_prot = prot.columns

    above_df2 = X_train.filter(items = above_idx, axis = 0)
    below_df2 = X_train.filter(items = below_idx, axis = 0)
    
    #calculate the anova on the train only - prot
    f_list=[]
    p_list=[]
    for i in above_df2.columns:
        F, p = f_oneway(above_df2.loc[:,i], below_df2.loc[:,i])
        f_list.append(F)
        p_list.append(p)

    threshold = 0.05/len(above_df.columns) 

    relevant=[]
    for i, z in enumerate(p_list):
        if z<threshold:
            relevant.append(i)
    print(len(relevant))

    # print the first few proteins that have been selected as different
    # make a table 
    df_anova2 = pd.DataFrame(np.array(p_list)[relevant], columns = ['p-values'], index = prot.columns[relevant])
    df_anova2['f-values'] = np.array(f_list)[relevant]
    we_want2 = df_anova2.sort_values(by='f-values', ascending = False)[0:200].index

    X_train_=X_train.loc[:, we_want2]
    X_test_ =X_test.loc[:, we_want2]
    print(X_train_.shape)
    print(X_test_.shape)

    scaler = StandardScaler()
    train_scaled = scaler.fit_transform(X_train_)
    test_scaled = scaler.fit_transform(X_test_)

    final_col2 = prot.columns[relevant]
    np.savetxt(str(seed)+'_1_featname.csv', we_want2, delimiter=',', fmt = '%s') 
    np.savetxt(str(seed)+'_1_tr.csv', train_scaled, delimiter=',')
    np.savetxt(str(seed)+'_1_te.csv', test_scaled, delimiter=',')
    
    # clin filtering
    clin_selected = pd.DataFrame(clin['age'])
    clin_selected['plt'] = clin['baseline_plt']
    clin_selected['wcc'] = clin['baseline_wcc']
    clin_selected['sbp'] = clin['admission_sbp']
    clin_selected['bd'] = clin['baseline_base_deficit']
    clin_selected = clin_selected.fillna(value = 0)
    clin_selected.shape
    col3 = np.array(clin_selected.columns)
    np.savetxt(str(seed)+'_3_featname.csv', col3, delimiter=',', fmt = '%s') 

    scaler = StandardScaler()
    clin_selected = scaler.fit_transform(clin_selected)
    
    
    X_train, X_test, y_train, y_test = train_test_split(clin_selected, y, random_state=seed)
    np.savetxt(str(seed)+'_3_tr.csv', X_train, delimiter=',')
    np.savetxt(str(seed)+'_3_te.csv', X_test, delimiter=',')
    
    np.savetxt(str(seed)+'labels_tr.csv', y_train, delimiter=',', fmt = '%s')
    np.savetxt(str(seed)+'labels_te.csv', y_test, delimiter=',', fmt = '%s')

[155, 157, 18, 163, 119]
4188
(309, 200)
(103, 200)
1848
(309, 200)
(103, 200)
3986
(309, 200)
(103, 200)
1443
(309, 200)
(103, 200)
3986
(309, 200)
(103, 200)
1009
(309, 200)
(103, 200)
3986
(309, 200)
(103, 200)
587
(309, 200)
(103, 200)
3986
(309, 200)
(103, 200)
2863
(309, 200)
(103, 200)


In [20]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

clf = RandomForestClassifier(max_depth=4, random_state=123)
clf.fit(X_train_, y_train)
preds = clf.predict(X_test_)
roc_auc_score(y_test, preds)

0.7580409356725146